# 🇬🇧 UK Retail Performance Hub
## Phase 1 | Notebook 1 of 4 — Data Ingestion

**What this notebook does:**
- Mounts your Google Drive
- Creates the project folder structure
- Downloads the UCI Online Retail dataset and ONS Retail Sales Index
- Validates both files are ready for cleaning

**Run order:** Run each cell top to bottom using the ▶ button or `Shift + Enter`

---

### Cell 1 — Mount Google Drive
This connects Colab to your Google Drive so files persist between sessions.
A popup will ask you to sign in and grant permission — click through and allow it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted successfully')

### Cell 2 — Set up project folder structure
Creates all the folders in your Google Drive that the project needs.

In [ ]:
import os
from pathlib import Path

# ── Root project path in your Google Drive ────────────────────────────────────
ROOT = Path('/content/drive/MyDrive/uk-retail-performance-hub')

# ── Create all folders ────────────────────────────────────────────────────────
folders = [
    ROOT / 'data' / 'raw',
    ROOT / 'data' / 'processed',
    ROOT / 'logs',
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print(f'  ✓ {folder}')

print('\n✓ Folder structure ready in Google Drive')

### Cell 3 — Install any missing libraries
Colab has most libraries pre-installed. This cell installs anything extra we need.

In [ ]:
!pip install openpyxl pyarrow --quiet
print('✓ Libraries ready')

### Cell 4 — Import libraries

In [ ]:
import requests
import pandas as pd
from pathlib import Path

RAW_DIR = ROOT / 'data' / 'raw'
print('✓ Imports complete')

### Cell 5 — Download UCI Online Retail Dataset

This is the main company transactions dataset (~500k rows, UK gift retailer 2010–2011).

> ⚠️ **If this download fails** (the UCI server can be slow):
> 1. Go to https://archive.ics.uci.edu/dataset/352/online+retail
> 2. Click Download → save the file as `online_retail.xlsx`
> 3. Upload it to your Drive at: `uk-retail-performance-hub/data/raw/online_retail.xlsx`
> 4. Skip to Cell 7

In [ ]:
uci_path = RAW_DIR / 'online_retail.xlsx'

if uci_path.exists():
    size_mb = uci_path.stat().st_size / 1_000_000
    print(f'✓ UCI file already exists ({size_mb:.1f} MB) — skipping download')
else:
    print('Downloading UCI Online Retail dataset...')
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'

    try:
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        uci_path.write_bytes(response.content)
        size_mb = uci_path.stat().st_size / 1_000_000
        print(f'✓ Downloaded and saved ({size_mb:.1f} MB)')
        print(f'  Saved to: {uci_path}')
    except Exception as e:
        print(f'✗ Download failed: {e}')
        print('')
        print('MANUAL STEPS:')
        print('  1. Go to https://archive.ics.uci.edu/dataset/352/online+retail')
        print('  2. Download the file and rename it online_retail.xlsx')
        print('  3. Upload to Google Drive at:')
        print(f'     {uci_path}')

### Cell 6 — Download ONS Retail Sales Index

This is the UK government's official monthly retail sales benchmark data.

> ⚠️ **If this download fails:**
> 1. Go to https://www.ons.gov.uk/businessindustryandtrade/retailindustry/datasets/retailsales
> 2. Download the time series CSV
> 3. Rename it `ons_rsi.csv` and upload to `uk-retail-performance-hub/data/raw/`

In [ ]:
ons_path = RAW_DIR / 'ons_rsi.csv'

if ons_path.exists():
    print(f'✓ ONS file already exists — skipping download')
else:
    print('Downloading ONS Retail Sales Index...')
    # ONS direct CSV export for retail sales all-retailing index
    url = 'https://www.ons.gov.uk/generator?format=csv&uri=/businessindustryandtrade/retailindustry/bulletins/retailsales/january2012/0'

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        ons_path.write_bytes(response.content)
        print(f'✓ ONS data saved to {ons_path}')
    except Exception as e:
        print(f'✗ Download failed: {e}')
        print('')
        print('MANUAL STEPS:')
        print('  1. Go to https://www.ons.gov.uk/businessindustryandtrade/retailindustry/datasets/retailsales')
        print('  2. Download the time series CSV')
        print('  3. Rename it ons_rsi.csv')
        print(f'  4. Upload to: {ons_path}')

### Cell 7 — Validate both files
Quick sanity check to confirm the files are readable and have the right columns.

In [ ]:
print('=' * 55)
print('  VALIDATION RESULTS')
print('=' * 55)

# ── Validate UCI ──────────────────────────────────────────────────────────────
print('\n[1] UCI Online Retail')
if uci_path.exists():
    try:
        df_check = pd.read_excel(uci_path, nrows=5)
        expected = {'InvoiceNo','StockCode','Description','Quantity',
                    'InvoiceDate','UnitPrice','CustomerID','Country'}
        missing = expected - set(df_check.columns)
        if missing:
            print(f'  ✗ Missing columns: {missing}')
        else:
            print(f'  ✓ File valid — columns: {list(df_check.columns)}')
            size_mb = uci_path.stat().st_size / 1_000_000
            print(f'  ✓ File size: {size_mb:.1f} MB')
    except Exception as e:
        print(f'  ✗ Could not read file: {e}')
else:
    print('  ✗ File not found — see manual download instructions above')

# ── Validate ONS ──────────────────────────────────────────────────────────────
print('\n[2] ONS Retail Sales Index')
if ons_path.exists():
    try:
        df_ons = pd.read_csv(ons_path, nrows=5, encoding='latin-1')
        print(f'  ✓ File valid — preview columns: {list(df_ons.columns[:4])}')
    except Exception as e:
        print(f'  ✗ Could not read file: {e}')
else:
    print('  ⚠ File not found — ONS data optional but recommended')

print('\n' + '=' * 55)
print('  ✓ Ingestion complete — run Notebook 02_clean.ipynb next')
print('=' * 55)